# Official Centralized LSTM

Ovaj notebook je sluzbeni execution path za centralized LSTM baseline.
Napisan je tako da se moze koristiti i kao demonstracijski materijal, a ne samo kao run script.

## Sto se ovdje dogada

Notebook prolazi kroz cijeli centralized LSTM workflow:

1. Ucitava official metadata i centralized `.npz` splitove.
2. Provjerava da se koristi isti task kao u ostatku rada.
3. Inicijalizira `LSTMRegressor`.
4. Trenera model kroz vise epoha uz validaciju i early stopping.
5. Racuna finalne metrike i sprema sve artefakte potrebne za eksperiment.

## Configuration

Ovdje su na jednom mjestu svi parametri koje je korisno mijenjati tijekom demonstracije ili novih runova.

In [1]:
import os
import random
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, TensorDataset

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "diplomski").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "diplomski").exists():
    raise RuntimeError("Run this notebook from the project root or from notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from diplomski.evaluation import (
    SUPPORTED_FEATURE_COUNT,
    build_prediction_frame,
    build_results_payload,
    compute_regression_metrics,
    configure_run_logger,
    load_experiment_metadata,
    make_run_name,
    save_predictions_csv,
    save_results_payload,
    validate_official_task,
    write_empty_communication_rounds,
)
from diplomski.models import LSTMRegressor
from diplomski.preprocessing.utils import save_json

RUN_NAME = os.environ.get("DIPLOMSKI_RUN_NAME", f"notebook_lstm_{make_run_name()}")
DATASET_DIR = Path(
    os.environ.get(
        "DIPLOMSKI_DATASET_DIR",
        str(PROJECT_ROOT / "data" / "processed" / "experiments" / "centralized"),
    )
).resolve()
METADATA_PATH = Path(
    os.environ.get(
        "DIPLOMSKI_EXPERIMENT_METADATA",
        str(PROJECT_ROOT / "data" / "processed" / "experiments" / "artifacts" / "experiment_metadata.json"),
    )
).resolve()
OUTPUT_ROOT = Path(
    os.environ.get(
        "DIPLOMSKI_OUTPUT_ROOT",
        str(PROJECT_ROOT / "data" / "processed" / "results" / "centralized" / "lstm"),
    )
).resolve()

EPOCHS = int(os.environ.get("DIPLOMSKI_LSTM_EPOCHS", "30"))
BATCH_SIZE = int(os.environ.get("DIPLOMSKI_LSTM_BATCH_SIZE", "256"))
HIDDEN_SIZE = int(os.environ.get("DIPLOMSKI_LSTM_HIDDEN_SIZE", "64"))
NUM_LAYERS = int(os.environ.get("DIPLOMSKI_LSTM_NUM_LAYERS", "1"))
DROPOUT = float(os.environ.get("DIPLOMSKI_LSTM_DROPOUT", "0.0"))
LR = float(os.environ.get("DIPLOMSKI_LSTM_LR", "0.001"))
WEIGHT_DECAY = float(os.environ.get("DIPLOMSKI_LSTM_WEIGHT_DECAY", "0.0"))
PATIENCE = int(os.environ.get("DIPLOMSKI_LSTM_PATIENCE", "5"))
GRAD_CLIP_NORM = float(os.environ.get("DIPLOMSKI_LSTM_GRAD_CLIP_NORM", "1.0"))
SEED = int(os.environ.get("DIPLOMSKI_LSTM_SEED", "42"))
DEVICE_REQUEST = os.environ.get("DIPLOMSKI_LSTM_DEVICE", "auto")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUN_NAME={RUN_NAME}")
print(f"DATASET_DIR={DATASET_DIR}")
print(f"METADATA_PATH={METADATA_PATH}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")
print(f"EPOCHS={EPOCHS} BATCH_SIZE={BATCH_SIZE} HIDDEN_SIZE={HIDDEN_SIZE}")


RuntimeError: Run this notebook from the project root or from notebooks/.

## Helper Functions: strukture podataka i loading

Prvo definiramo male strukture i funkcije za ucitavanje `.npz` splitova.
Ove provjere su vazne kako bi se rano uhvatile greske u obliku podataka, broju uzoraka ili sadrzaju splitova.

In [2]:
@dataclass(frozen=True)
class SplitArrays:
    X: np.ndarray
    y: np.ndarray
    t: np.ndarray
    station_ids: np.ndarray


@dataclass(frozen=True)
class CentralizedDatasetBundle:
    train: SplitArrays
    val: SplitArrays
    test: SplitArrays


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def select_device(device_name: str) -> torch.device:
    normalized = device_name.strip().lower()
    if normalized == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if normalized == "cpu":
        return torch.device("cpu")
    if normalized == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is not available.")
        return torch.device("cuda")
    raise ValueError(f"Unsupported device '{device_name}'. Use auto, cpu, or cuda.")


def load_split_npz(
    split_path: Path,
    split_name: str,
    *,
    expected_window_size: int,
    expected_feature_count: int,
) -> SplitArrays:
    if not split_path.exists():
        raise FileNotFoundError(f"Missing centralized split file: {split_path}")

    with np.load(split_path) as data:
        required_keys = {"X", "y", "t", "station_ids"}
        if set(data.files) != required_keys:
            raise RuntimeError(
                f"{split_name} split must contain keys {sorted(required_keys)}, got {sorted(data.files)}."
            )
        X = data["X"].astype(np.float32, copy=False)
        y = data["y"].astype(np.float32, copy=False)
        t = data["t"].astype(str)
        station_ids = data["station_ids"].astype(str)

    if X.ndim != 3:
        raise RuntimeError(f"{split_name} X must be 3D, got shape {X.shape}.")
    if y.ndim != 1:
        raise RuntimeError(f"{split_name} y must be 1D, got shape {y.shape}.")
    if X.shape[0] != y.shape[0] or X.shape[0] != t.shape[0] or X.shape[0] != station_ids.shape[0]:
        raise RuntimeError(f"{split_name} arrays must agree on sample count.")
    if X.shape[1] != expected_window_size:
        raise RuntimeError(
            f"{split_name} X has window size {X.shape[1]}, expected {expected_window_size}."
        )
    if X.shape[2] != expected_feature_count:
        raise RuntimeError(
            f"{split_name} X has feature count {X.shape[2]}, expected {expected_feature_count}."
        )
    if X.shape[0] == 0:
        raise RuntimeError(f"{split_name} split is empty.")
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise RuntimeError(f"{split_name} split contains NaN or Inf values.")

    return SplitArrays(X=X, y=y, t=t, station_ids=station_ids)


def load_centralized_dataset(dataset_dir: Path, metadata: dict[str, Any]) -> CentralizedDatasetBundle:
    expected_window_size = int(metadata["window_size"])
    expected_feature_count = int(metadata["feature_count"])
    splits = {
        split_name: load_split_npz(
            dataset_dir / f"{split_name}.npz",
            split_name,
            expected_window_size=expected_window_size,
            expected_feature_count=expected_feature_count,
        )
        for split_name in ("train", "val", "test")
    }

    metadata_counts = metadata.get("centralized_counts", {})
    for split_name, split_arrays in splits.items():
        expected_count = int(metadata_counts[split_name])
        actual_count = int(split_arrays.X.shape[0])
        if actual_count != expected_count:
            raise RuntimeError(
                f"{split_name} split count mismatch: expected {expected_count}, got {actual_count}."
            )

    return CentralizedDatasetBundle(
        train=splits["train"],
        val=splits["val"],
        test=splits["test"],
    )


## Helper Functions: DataLoader, trening i evaluacija

Ovdje su funkcije koje rade stvarni PyTorch posao: batchanje, trening jedne epohe, evaluaciju i spremanje history/config artefakata.

In [3]:
def build_dataloader(
    split_arrays: SplitArrays,
    batch_size: int,
    *,
    shuffle: bool,
    device: torch.device,
) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(split_arrays.X), torch.from_numpy(split_arrays.y))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=device.type == "cuda",
    )


def train_one_epoch(
    *,
    model: nn.Module,
    split_arrays: SplitArrays,
    batch_size: int,
    device: torch.device,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    grad_clip_norm: float,
) -> float:
    model.train()
    loader = build_dataloader(split_arrays, batch_size, shuffle=True, device=device)
    total_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
        y_batch = y_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")

        optimizer.zero_grad(set_to_none=True)
        predictions = model(X_batch)
        if not torch.isfinite(predictions).all():
            raise RuntimeError("Non-finite predictions encountered during training.")

        loss = loss_fn(predictions, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss encountered during training.")

        loss.backward()
        if grad_clip_norm > 0.0:
            clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()

        batch_size_actual = int(y_batch.shape[0])
        total_loss += float(loss.item()) * batch_size_actual
        total_samples += batch_size_actual

    return total_loss / total_samples


def evaluate_split(
    *,
    model: nn.Module,
    split_arrays: SplitArrays,
    batch_size: int,
    device: torch.device,
    loss_fn: nn.Module,
    method_name: str,
) -> dict[str, Any]:
    model.eval()
    loader = build_dataloader(split_arrays, batch_size, shuffle=False, device=device)
    total_loss = 0.0
    total_samples = 0
    y_predictions: list[np.ndarray] = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")
            y_batch = y_batch.to(device=device, dtype=torch.float32, non_blocking=device.type == "cuda")

            predictions = model(X_batch)
            if not torch.isfinite(predictions).all():
                raise RuntimeError("Non-finite predictions encountered during evaluation.")

            loss = loss_fn(predictions, y_batch)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered during evaluation.")

            batch_size_actual = int(y_batch.shape[0])
            total_loss += float(loss.item()) * batch_size_actual
            total_samples += batch_size_actual
            y_predictions.append(predictions.cpu().numpy())

    y_pred = np.concatenate(y_predictions, axis=0)
    metrics = compute_regression_metrics(split_arrays.y, y_pred)
    prediction_frame = build_prediction_frame(
        t=split_arrays.t,
        station_ids=split_arrays.station_ids,
        y_true=split_arrays.y,
        y_pred=y_pred,
        method_name=method_name,
    )

    return {
        "loss": total_loss / total_samples,
        "metrics": metrics,
        "prediction_frame": prediction_frame,
    }


def count_trainable_parameters(model: nn.Module) -> int:
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


def estimate_model_state_bytes(model: nn.Module) -> int:
    state_dict = model.state_dict()
    return int(sum(tensor.numel() * tensor.element_size() for tensor in state_dict.values()))


def save_history_csv(history_rows: list[dict[str, Any]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history_rows).to_csv(output_path, index=False)


def save_training_config(output_path: Path, metadata: dict[str, Any], selected_device: torch.device) -> None:
    payload = {
        "dataset_dir": str(DATASET_DIR),
        "experiment_metadata_path": str(METADATA_PATH),
        "output_dir": str(OUTPUT_ROOT),
        "run_name": RUN_NAME,
        "device": str(selected_device),
        "seed": SEED,
        "model": {
            "input_size": int(metadata["feature_count"]),
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
        },
        "optimization": {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "grad_clip_norm": GRAD_CLIP_NORM,
        },
        "task": {
            "window_size": int(metadata["window_size"]),
            "horizon": int(metadata["horizon"]),
            "target_column": metadata["target_column"],
            "feature_count": int(metadata["feature_count"]),
        },
    }
    save_json(payload, output_path)


## 1. Metadata i sluzbeni task

Ovdje potvrdujemo da notebook koristi isti canonical forecasting task kao i ostatak projekta.

In [4]:
metadata = load_experiment_metadata(METADATA_PATH)
validate_official_task(metadata, expected_feature_count=SUPPORTED_FEATURE_COUNT)
dataset = load_centralized_dataset(DATASET_DIR, metadata)
selected_device = select_device(DEVICE_REQUEST)
set_global_seed(SEED)

run_dir = OUTPUT_ROOT / RUN_NAME
logger = configure_run_logger(
    name=f"diplomski.notebooks.lstm.{RUN_NAME}",
    log_path=run_dir / "run.log",
)
save_training_config(run_dir / "config.json", metadata, selected_device)

pd.DataFrame(
    [
        {"field": "selected_device", "value": str(selected_device)},
        {"field": "window_size", "value": metadata["window_size"]},
        {"field": "horizon", "value": metadata["horizon"]},
        {"field": "target_column", "value": metadata["target_column"]},
        {"field": "feature_count", "value": metadata["feature_count"]},
    ]
)


,field,value
0,selected_device,cpu
1,window_size,24
2,horizon,1
3,target_column,volume
4,feature_count,19


## 2. Pregled ulaznih splitova

Model ne radi direktno nad hourly CSV datotekama, nego nad vec pripremljenim centralized `.npz` splitovima.
Svaki uzorak ima oblik `[24, 19]`, odnosno 24 sata povijesti i 19 featurea po satu.

In [5]:
dataset_overview = pd.DataFrame(
    [
        {"split": "train", "X_shape": dataset.train.X.shape, "y_shape": dataset.train.y.shape},
        {"split": "val", "X_shape": dataset.val.X.shape, "y_shape": dataset.val.y.shape},
        {"split": "test", "X_shape": dataset.test.X.shape, "y_shape": dataset.test.y.shape},
    ]
)
dataset_overview


,split,X_shape,y_shape
0,train,"(110642, 24, 19)","(110642,)"
1,val,"(6824, 24, 19)","(6824,)"
2,test,"(30275, 24, 19)","(30275,)"


### Primjer jednog mini-batcha

Prije treninga je korisno provjeriti kako izgleda jedan batch koji ulazi u model.

In [6]:
example_loader = build_dataloader(dataset.train, batch_size=min(BATCH_SIZE, 64), shuffle=True, device=selected_device)
example_X, example_y = next(iter(example_loader))

pd.DataFrame(
    [
        {"item": "mini_batch_X_shape", "value": tuple(example_X.shape)},
        {"item": "mini_batch_y_shape", "value": tuple(example_y.shape)},
        {"item": "single_sequence_shape", "value": tuple(example_X[0].shape)},
    ]
)


,item,value
0,mini_batch_X_shape,"(64, 24, 19)"
1,mini_batch_y_shape,"(64,)"
2,single_sequence_shape,"(24, 19)"


## 3. Inicijalizacija modela

`LSTMRegressor` koristi zadnji hidden state kao reprezentaciju cijelog prozora, a zatim linearni sloj vraca jednu skalarni predikciju.

In [7]:
model = LSTMRegressor(
    input_size=int(metadata["feature_count"]),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(selected_device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()
parameter_count = count_trainable_parameters(model)
model_state_bytes = estimate_model_state_bytes(model)
best_model_path = run_dir / "best_model.pt"

logger.info(
    "Device=%s | parameter_count=%d | model_state_bytes=%d",
    selected_device,
    parameter_count,
    model_state_bytes,
)

pd.DataFrame(
    [
        {"item": "parameter_count", "value": parameter_count},
        {"item": "model_state_bytes", "value": model_state_bytes},
        {"item": "hidden_size", "value": HIDDEN_SIZE},
        {"item": "num_layers", "value": NUM_LAYERS},
        {"item": "dropout", "value": DROPOUT},
    ]
)


2026-04-07 21:09:56,254 | INFO | Device=cpu | parameter_count=21825 | model_state_bytes=87300


,item,value
0,parameter_count,21825.0
1,model_state_bytes,87300.0
2,hidden_size,64.0
3,num_layers,1.0
4,dropout,0.0


## 4. Trening loop

U svakoj epohi radimo:
- trening nad train splitom
- evaluaciju nad validation splitom
- spremanje najboljeg checkpointa po `val_rmse`
- early stopping ako se validation ne popravlja dovoljno dugo

In [8]:
history_rows: list[dict[str, Any]] = []
best_val_rmse = float("inf")
best_epoch: int | None = None
epochs_without_improvement = 0
early_stopped = False

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        split_arrays=dataset.train,
        batch_size=BATCH_SIZE,
        device=selected_device,
        optimizer=optimizer,
        loss_fn=loss_fn,
        grad_clip_norm=GRAD_CLIP_NORM,
    )
    val_eval = evaluate_split(
        model=model,
        split_arrays=dataset.val,
        batch_size=BATCH_SIZE,
        device=selected_device,
        loss_fn=loss_fn,
        method_name="lstm",
    )

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_eval["loss"]),
            "val_rmse": float(val_eval["metrics"]["rmse"]),
            "val_mae": float(val_eval["metrics"]["mae"]),
            "val_r2": float(val_eval["metrics"]["r2"]),
        }
    )
    logger.info(
        "Epoch %d/%d | train_loss=%.6f | val_loss=%.6f | val_rmse=%.6f | val_mae=%.6f | val_r2=%.6f",
        epoch,
        EPOCHS,
        train_loss,
        val_eval["loss"],
        val_eval["metrics"]["rmse"],
        val_eval["metrics"]["mae"],
        val_eval["metrics"]["r2"],
    )

    if float(val_eval["metrics"]["rmse"]) < best_val_rmse:
        best_val_rmse = float(val_eval["metrics"]["rmse"])
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
        logger.info("Saved new best model at epoch %d with val_rmse=%.6f", epoch, best_val_rmse)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            early_stopped = True
            logger.info("Early stopping triggered at epoch %d.", epoch)
            break

if best_epoch is None:
    raise RuntimeError("Training finished without producing a valid checkpoint.")


KeyboardInterrupt: 

### History po epohama

Ova tablica je korisna za brzu procjenu konvergencije tijekom demonstracije.

In [ ]:
history_df = pd.DataFrame(history_rows)
history_df


## 5. Finalna evaluacija najboljeg modela

Nakon treninga ponovno ucitavamo najbolji checkpoint i tek tada racunamo finalne metrike za train, val i test.

In [ ]:
best_state = torch.load(best_model_path, map_location=selected_device)
model.load_state_dict(best_state)

train_eval = evaluate_split(
    model=model,
    split_arrays=dataset.train,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name="lstm",
)
val_eval = evaluate_split(
    model=model,
    split_arrays=dataset.val,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name="lstm",
)
test_eval = evaluate_split(
    model=model,
    split_arrays=dataset.test,
    batch_size=BATCH_SIZE,
    device=selected_device,
    loss_fn=loss_fn,
    method_name="lstm",
)

metrics_summary = pd.DataFrame(
    [
        {"split": "train", **train_eval["metrics"]},
        {"split": "val", **val_eval["metrics"]},
        {"split": "test", **test_eval["metrics"]},
    ]
)
metrics_summary


### Kratki pogled u test predikcije

Isti prediction schema koristi se i za naive baselinee, pa su rezultati direktno usporedivi.

In [ ]:
test_eval["prediction_frame"].head()


## 6. Spremanje artefakata

Run zapisujemo u isti canonical format kao i ostale eksperimente.
Za LSTM dodatno spremamo checkpoint, history i config.

In [ ]:
split_counts = {
    "train": int(dataset.train.X.shape[0]),
    "val": int(dataset.val.X.shape[0]),
    "test": int(dataset.test.X.shape[0]),
}
results_payload = build_results_payload(
    run_name=RUN_NAME,
    stage="baseline",
    scenario="centralized",
    method="lstm",
    generated_at_utc=datetime.now(timezone.utc).isoformat(),
    epochs_completed=len(history_rows),
    best_epoch=best_epoch,
    target_column=str(metadata["target_column"]),
    window_size=int(metadata["window_size"]),
    horizon=int(metadata["horizon"]),
    experiment_metadata_path=METADATA_PATH,
    valid_station_ids=list(metadata["valid_station_ids"]),
    split_counts=split_counts,
    metrics_by_split={
        "train": train_eval["metrics"],
        "val": val_eval["metrics"],
        "test": test_eval["metrics"],
    },
    training_info={
        "trained": True,
        "early_stopped": early_stopped,
        "best_val_rmse": float(best_val_rmse),
        "parameter_count": parameter_count,
    },
    communication_info={
        "communication_enabled": False,
        "logical_client_count": int(metadata["valid_station_count"]),
        "round_count": 0,
        "messages_up": 0,
        "messages_down": 0,
        "payload_bytes_up": 0,
        "payload_bytes_down": 0,
        "payload_bytes_total": 0,
        "model_state_bytes": model_state_bytes,
    },
    dataset_dir=DATASET_DIR,
    hourly_dir=None,
)

save_results_payload(results_payload, run_dir / "results.json")
save_predictions_csv(test_eval["prediction_frame"], run_dir / "test_predictions.csv")
write_empty_communication_rounds(run_dir / "communication_rounds.csv")
save_history_csv(history_rows, run_dir / "history.csv")

logger.info(
    "Completed centralized LSTM run | best_epoch=%d | test_rmse=%.6f | test_mae=%.6f | test_r2=%.6f",
    best_epoch,
    test_eval["metrics"]["rmse"],
    test_eval["metrics"]["mae"],
    test_eval["metrics"]["r2"],
)
logger.info("Artifacts saved to %s", run_dir)

pd.DataFrame(
    [
        {"artifact": artifact_path.name, "path": str(artifact_path)}
        for artifact_path in sorted(run_dir.iterdir())
    ]
)
